In [ ]:
# 必要なライブラリのインストール
!pip install openai numpy pandas scikit-learn

In [ ]:
# RAG検索精度評価システム（300件データ版）

import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Tuple
import time
from google.colab import userdata

# OpenAIクライアントの初期化
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def generate_test_data():
    """300件のテストデータを生成"""
    qa_data = []

    # カテゴリー1: 機械学習・AI（30件）
    ml_topics = [
        ("教師あり学習", "ラベル付きデータを使用してモデルを訓練する機械学習の手法"),
        ("教師なし学習", "ラベルなしデータからパターンを発見する機械学習の手法"),
        ("半教師あり学習", "少量のラベル付きデータと大量のラベルなしデータを組み合わせる学習手法"),
        ("深層学習", "多層のニューラルネットワークを使用する機械学習の一種"),
        ("強化学習", "エージェントが環境と相互作用しながら報酬を最大化する学習手法"),
        ("転移学習", "事前学習済みモデルを新しいタスクに適用する手法"),
        ("連合学習", "データを中央に集めずに分散環境でモデルを学習する手法"),
        ("メタ学習", "学習方法自体を学習する機械学習のアプローチ"),
        ("敵対的学習", "敵対的な例を使用してモデルの頑健性を向上させる手法"),
        ("アンサンブル学習", "複数のモデルを組み合わせて予測精度を向上させる手法"),
        ("オンライン学習", "データを順次処理しながらモデルを更新する学習方法"),
        ("バッチ学習", "データセット全体を使用してモデルを訓練する学習方法"),
        ("勾配降下法", "損失関数を最小化するための最適化アルゴリズム"),
        ("誤差逆伝播法", "ニューラルネットワークの学習に使用される勾配計算手法"),
        ("過学習", "モデルが訓練データに過度に適合し汎化性能が低下する現象"),
        ("正則化", "過学習を防ぐためにモデルの複雑さを制限する手法"),
        ("交差検証", "モデルの汎化性能を評価するためのデータ分割手法"),
        ("特徴量エンジニアリング", "機械学習モデルの入力となる特徴量を設計・選択するプロセス"),
        ("次元削減", "高次元データを低次元に変換する手法"),
        ("クラスタリング", "データを類似性に基づいてグループ分けする教師なし学習手法"),
        ("分類問題", "データを事前定義されたカテゴリーに分類するタスク"),
        ("回帰問題", "連続値を予測する機械学習のタスク"),
        ("異常検知", "正常パターンから逸脱したデータを検出する手法"),
        ("時系列予測", "過去のデータから将来の値を予測するタスク"),
        ("自然言語処理", "コンピュータが人間の言語を理解・生成する技術"),
        ("コンピュータビジョン", "画像や動画からの情報抽出を行う技術"),
        ("音声認識", "音声信号をテキストに変換する技術"),
        ("推薦システム", "ユーザーの好みに基づいてアイテムを推薦するシステム"),
        ("生成AI", "新しいコンテンツを生成する人工知能技術"),
        ("説明可能AI", "AIの意思決定プロセスを人間が理解できるようにする技術")
    ]

    for topic, answer in ml_topics:
        qa_data.append({
            "category": "機械学習・AI",
            "question": f"{topic}とは何ですか？",
            "answer": f"{topic}は、{answer}です。"
        })

    # カテゴリー2: プログラミング言語（30件）
    prog_langs = [
        ("Python", "汎用的で読みやすい高水準プログラミング言語", "データサイエンスやAI開発"),
        ("JavaScript", "Web開発の中心的な言語", "フロントエンドとバックエンド開発"),
        ("Java", "プラットフォーム独立な言語", "エンタープライズアプリケーション"),
        ("C++", "高性能なシステムプログラミング言語", "ゲーム開発やシステムソフトウェア"),
        ("C#", "Microsoftが開発した言語", ".NETフレームワークでの開発"),
        ("Go", "Googleが開発した言語", "並行処理とクラウドネイティブ開発"),
        ("Rust", "メモリ安全性を重視した言語", "システムプログラミング"),
        ("TypeScript", "JavaScriptの型付き拡張", "大規模なWebアプリケーション"),
        ("Swift", "Appleが開発した言語", "iOSとmacOSアプリケーション"),
        ("Kotlin", "JVM上で動作する言語", "Androidアプリケーション開発"),
        ("R", "統計解析に特化した言語", "データ分析と統計計算"),
        ("PHP", "サーバーサイドWeb開発言語", "動的なWebページ生成"),
        ("Ruby", "開発者の幸福を重視した言語", "Webアプリケーション開発"),
        ("Scala", "関数型とオブジェクト指向を統合", "ビッグデータ処理"),
        ("Perl", "テキスト処理に強い言語", "システム管理とWeb開発"),
        ("Haskell", "純粋関数型プログラミング言語", "研究と教育"),
        ("Clojure", "JVM上で動作するLisp方言", "並行処理とデータ処理"),
        ("Elixir", "Erlang VM上で動作する言語", "高可用性システム"),
        ("Julia", "科学計算向け高性能言語", "数値計算と研究"),
        ("Dart", "Googleが開発した言語", "Flutterによるクロスプラットフォーム開発"),
        ("Lua", "軽量な組み込み言語", "ゲームスクリプティング"),
        ("MATLAB", "数値計算環境と言語", "工学と科学計算"),
        ("Fortran", "科学計算の歴史的言語", "数値シミュレーション"),
        ("COBOL", "ビジネス向けプログラミング言語", "レガシーシステム"),
        ("Assembly", "低レベルプログラミング言語", "ハードウェア制御"),
        ("SQL", "データベースクエリ言語", "データベース操作"),
        ("Bash", "Unixシェルスクリプト言語", "システム自動化"),
        ("PowerShell", "Windowsの管理用言語", "システム管理とタスク自動化"),
        ("Objective-C", "Appleの旧主力言語", "レガシーiOSアプリ"),
        ("Visual Basic", "Microsoftの初心者向け言語", "Windows アプリケーション")
    ]

    for lang, desc, use in prog_langs:
        qa_data.append({
            "category": "プログラミング言語",
            "question": f"{lang}の特徴は何ですか？",
            "answer": f"{lang}は{desc}で、主に{use}に使用されます。"
        })

    # カテゴリー3: データベース（30件）
    db_topics = [
        ("リレーショナルデータベース", "表形式でデータを管理し、SQLでアクセスする"),
        ("NoSQLデータベース", "非リレーショナルな構造でデータを管理する"),
        ("インメモリデータベース", "メインメモリ上でデータを管理し高速アクセスを実現"),
        ("グラフデータベース", "ノードとエッジでデータの関係性を表現"),
        ("時系列データベース", "時間軸に沿ったデータの保存と検索に特化"),
        ("キーバリューストア", "シンプルなキーと値のペアでデータを管理"),
        ("ドキュメントデータベース", "JSONのような文書形式でデータを保存"),
        ("カラムファミリーデータベース", "列指向でデータを格納し大規模データに対応"),
        ("オブジェクトデータベース", "オブジェクト指向のデータモデルを直接サポート"),
        ("分散データベース", "複数のノードにデータを分散して管理"),
        ("MySQL", "オープンソースの人気リレーショナルデータベース"),
        ("PostgreSQL", "高機能なオープンソースリレーショナルデータベース"),
        ("MongoDB", "人気のドキュメント指向NoSQLデータベース"),
        ("Redis", "高速なインメモリキーバリューストア"),
        ("Cassandra", "分散型カラムファミリーデータベース"),
        ("Oracle Database", "エンタープライズ向け商用データベース"),
        ("SQL Server", "Microsoftの商用リレーショナルデータベース"),
        ("SQLite", "軽量な組み込み用データベース"),
        ("Neo4j", "代表的なグラフデータベース"),
        ("Elasticsearch", "全文検索と分析に特化したデータストア"),
        ("DynamoDB", "AWSのマネージドNoSQLデータベース"),
        ("CouchDB", "RESTful APIを持つドキュメントデータベース"),
        ("InfluxDB", "時系列データ専用のデータベース"),
        ("MariaDB", "MySQLから派生したデータベース"),
        ("HBase", "Hadoopエコシステムの分散データベース"),
        ("データベース正規化", "データの冗長性を減らす設計手法"),
        ("ACID特性", "トランザクションの信頼性を保証する特性"),
        ("CAP定理", "分散システムの一貫性、可用性、分断耐性のトレードオフ"),
        ("インデックス", "データベースの検索性能を向上させる仕組み"),
        ("レプリケーション", "データの複製を作成して可用性を向上させる")
    ]

    for topic, desc in db_topics:
        qa_data.append({
            "category": "データベース",
            "question": f"{topic}とは何ですか？",
            "answer": f"{topic}は、{desc}データベース技術です。"
        })

    # カテゴリー4: クラウドサービス（30件）
    cloud_topics = [
        ("AWS", "Amazon Web Services - Amazonの包括的クラウドプラットフォーム"),
        ("Microsoft Azure", "Microsoftのエンタープライズ向けクラウドサービス"),
        ("Google Cloud Platform", "Googleの機械学習に強いクラウドサービス"),
        ("IaaS", "Infrastructure as a Service - インフラをサービスとして提供"),
        ("PaaS", "Platform as a Service - 開発プラットフォームを提供"),
        ("SaaS", "Software as a Service - ソフトウェアをサービスとして提供"),
        ("サーバーレス", "サーバー管理不要でコードを実行できるアーキテクチャ"),
        ("コンテナ", "アプリケーションと依存関係をパッケージ化する技術"),
        ("マイクロサービス", "アプリケーションを小さなサービスに分割するアーキテクチャ"),
        ("オートスケーリング", "負荷に応じて自動的にリソースを調整する機能"),
        ("ロードバランサー", "トラフィックを複数のサーバーに分散する仕組み"),
        ("CDN", "Content Delivery Network - コンテンツを地理的に分散配信"),
        ("VPC", "Virtual Private Cloud - 仮想プライベートネットワーク"),
        ("EC2", "AWS の仮想サーバーサービス"),
        ("S3", "AWS のオブジェクトストレージサービス"),
        ("Lambda", "AWS のサーバーレスコンピューティングサービス"),
        ("Kubernetes", "コンテナオーケストレーションプラットフォーム"),
        ("Docker", "コンテナ化技術のデファクトスタンダード"),
        ("CI/CD", "継続的インテグレーション/デプロイメント"),
        ("DevOps", "開発と運用を統合するカルチャーと実践"),
        ("マルチクラウド", "複数のクラウドプロバイダーを使用する戦略"),
        ("ハイブリッドクラウド", "オンプレミスとクラウドを組み合わせる"),
        ("エッジコンピューティング", "データ発生源に近い場所で処理を行う"),
        ("クラウドネイティブ", "クラウド環境に最適化されたアプリケーション設計"),
        ("Infrastructure as Code", "インフラをコードで管理する手法"),
        ("監視とロギング", "システムの健全性を追跡する仕組み"),
        ("バックアップとディザスタリカバリ", "データ保護と災害復旧"),
        ("クラウドセキュリティ", "クラウド環境のセキュリティ対策"),
        ("コスト最適化", "クラウドリソースのコストを効率化"),
        ("API Gateway", "APIの統一的な入口を提供するサービス")
    ]

    for topic, desc in cloud_topics:
        qa_data.append({
            "category": "クラウドサービス",
            "question": f"{topic}について説明してください",
            "answer": f"{topic}は、{desc}する技術・サービスです。"
        })

    # カテゴリー5: Web開発（30件）
    web_topics = [
        ("HTML", "Webページの構造を定義するマークアップ言語"),
        ("CSS", "Webページのスタイルを定義するスタイルシート言語"),
        ("React", "Facebookが開発したJavaScriptライブラリ"),
        ("Angular", "Googleが開発したWebアプリケーションフレームワーク"),
        ("Vue.js", "プログレッシブJavaScriptフレームワーク"),
        ("Node.js", "サーバーサイドJavaScript実行環境"),
        ("Express.js", "Node.js用の軽量Webフレームワーク"),
        ("REST API", "RESTful原則に基づくAPI設計"),
        ("GraphQL", "APIのクエリ言語とランタイム"),
        ("WebSocket", "双方向リアルタイム通信プロトコル"),
        ("PWA", "Progressive Web App - ネイティブアプリ的なWeb"),
        ("SPA", "Single Page Application - 単一ページアプリケーション"),
        ("SSR", "Server-Side Rendering - サーバーサイドレンダリング"),
        ("SSG", "Static Site Generation - 静的サイト生成"),
        ("JAMstack", "JavaScript, APIs, Markupのアーキテクチャ"),
        ("Webpack", "モジュールバンドラー"),
        ("レスポンシブデザイン", "様々なデバイスに対応するデザイン"),
        ("アクセシビリティ", "すべての人がWebを利用できるようにする"),
        ("SEO", "Search Engine Optimization - 検索エンジン最適化"),
        ("HTTPS", "セキュアなHTTP通信プロトコル"),
        ("CORS", "Cross-Origin Resource Sharing - オリジン間リソース共有"),
        ("JWT", "JSON Web Token - 認証トークン形式"),
        ("OAuth", "認可のためのオープンスタンダード"),
        ("WebAssembly", "ブラウザで高速実行可能なバイナリ形式"),
        ("Service Worker", "バックグラウンドで動作するスクリプト"),
        ("Web Components", "再利用可能なカスタム要素を作成する技術"),
        ("CSS Grid", "2次元レイアウトシステム"),
        ("Flexbox", "柔軟なボックスレイアウトモデル"),
        ("Sass/SCSS", "CSSプリプロセッサ"),
        ("TypeScript", "型付きJavaScript")
    ]

    for topic, desc in web_topics:
        qa_data.append({
            "category": "Web開発",
            "question": f"{topic}とは何ですか？",
            "answer": f"{topic}は、{desc}です。"
        })

    # カテゴリー6: セキュリティ（30件）
    security_topics = [
        ("暗号化", "データを読めない形式に変換して保護する技術"),
        ("認証", "ユーザーの身元を確認するプロセス"),
        ("認可", "リソースへのアクセス権限を管理するプロセス"),
        ("ファイアウォール", "ネットワークトラフィックを制御するセキュリティシステム"),
        ("VPN", "Virtual Private Network - 安全な通信トンネル"),
        ("SSL/TLS", "安全な通信を確立するプロトコル"),
        ("二要素認証", "2つの異なる要素で認証を行う方法"),
        ("ペネトレーションテスト", "システムの脆弱性を発見するテスト"),
        ("脆弱性スキャン", "既知の脆弱性を自動検出する"),
        ("DDoS攻撃", "分散サービス拒否攻撃"),
        ("SQLインジェクション", "SQLクエリを悪用する攻撃"),
        ("XSS", "Cross-Site Scripting - スクリプト挿入攻撃"),
        ("CSRF", "Cross-Site Request Forgery - リクエスト偽造攻撃"),
        ("ゼロデイ攻撃", "未知の脆弱性を悪用する攻撃"),
        ("ランサムウェア", "データを暗号化して身代金を要求するマルウェア"),
        ("フィッシング", "偽装して情報を盗む詐欺手法"),
        ("ソーシャルエンジニアリング", "人間の心理を悪用する攻撃"),
        ("IDS/IPS", "侵入検知・防止システム"),
        ("SIEM", "セキュリティ情報イベント管理"),
        ("エンドポイントセキュリティ", "端末レベルのセキュリティ対策"),
        ("ゼロトラストセキュリティ", "すべてを信頼しないセキュリティモデル"),
        ("セキュリティパッチ", "脆弱性を修正するアップデート"),
        ("暗号化アルゴリズム", "データを暗号化する数学的手法"),
        ("デジタル証明書", "身元を証明する電子的な証明書"),
        ("セキュアコーディング", "安全なコードを書く実践"),
        ("セキュリティ監査", "セキュリティ対策の評価"),
        ("インシデントレスポンス", "セキュリティ事故への対応"),
        ("データ漏洩", "機密情報の不正な流出"),
        ("コンプライアンス", "規制や基準への準拠"),
        ("GDPR", "EU一般データ保護規則")
    ]

    for topic, desc in security_topics:
        qa_data.append({
            "category": "セキュリティ",
            "question": f"{topic}について教えてください",
            "answer": f"{topic}は、{desc}するセキュリティ関連の概念です。"
        })

    # カテゴリー7: データサイエンス（30件）
    ds_topics = [
        ("データ分析", "データから有意義な洞察を抽出するプロセス"),
        ("データマイニング", "大量のデータからパターンを発見する技術"),
        ("ビッグデータ", "従来の手法では扱えない大規模データ"),
        ("データビジュアライゼーション", "データを視覚的に表現する技術"),
        ("統計分析", "統計的手法を用いたデータ分析"),
        ("予測分析", "過去のデータから将来を予測する分析"),
        ("A/Bテスト", "2つのバージョンを比較する実験手法"),
        ("データクリーニング", "データの品質を向上させる前処理"),
        ("特徴量選択", "モデルに使用する変数を選択するプロセス"),
        ("相関分析", "変数間の関係性を分析する手法"),
        ("回帰分析", "変数間の関係をモデル化する統計手法"),
        ("分類分析", "データをカテゴリーに分類する分析"),
        ("クラスター分析", "データをグループに分ける分析手法"),
        ("時系列分析", "時間軸に沿ったデータの分析"),
        ("主成分分析", "データの次元を削減する手法"),
        ("決定木", "ツリー構造で意思決定を表現するモデル"),
        ("ランダムフォレスト", "複数の決定木を組み合わせたモデル"),
        ("サポートベクターマシン", "分類と回帰のための機械学習手法"),
        ("ニューラルネットワーク", "脳の構造を模倣した計算モデル"),
        ("k-means", "代表的なクラスタリングアルゴリズム"),
        ("ベイズ統計", "ベイズの定理に基づく統計手法"),
        ("探索的データ分析", "データの特性を理解する初期分析"),
        ("データウェアハウス", "分析用に最適化されたデータ保管庫"),
        ("ETL", "Extract, Transform, Load - データ処理パイプライン"),
        ("データレイク", "生データを保存する大規模リポジトリ"),
        ("BI", "Business Intelligence - ビジネスインテリジェンス"),
        ("データガバナンス", "データの管理と品質を保証する枠組み"),
        ("データサイエンティスト", "データから価値を創出する専門家"),
        ("機械学習エンジニア", "機械学習システムを構築・運用する専門家"),
        ("データパイプライン", "データ処理の自動化フロー")
    ]

    for topic, desc in ds_topics:
        qa_data.append({
            "category": "データサイエンス",
            "question": f"{topic}とは何ですか？",
            "answer": f"{topic}は、{desc}です。"
        })

    # カテゴリー8: DevOps（30件）
    devops_topics = [
        ("DevOps", "開発と運用を統合する文化と実践"),
        ("CI/CD", "継続的インテグレーションと継続的デリバリー"),
        ("Jenkins", "人気のあるCI/CDツール"),
        ("GitLab CI", "GitLabに統合されたCI/CDプラットフォーム"),
        ("GitHub Actions", "GitHubのワークフロー自動化ツール"),
        ("Docker", "コンテナ化技術のスタンダード"),
        ("Kubernetes", "コンテナオーケストレーションプラットフォーム"),
        ("Helm", "Kubernetesのパッケージマネージャー"),
        ("Terraform", "Infrastructure as Codeツール"),
        ("Ansible", "構成管理と自動化ツール"),
        ("監視", "システムの状態を継続的に追跡する"),
        ("ロギング", "システムイベントを記録・分析する"),
        ("Prometheus", "オープンソースの監視システム"),
        ("Grafana", "メトリクスの可視化ツール"),
        ("ELK Stack", "Elasticsearch, Logstash, Kibanaの組み合わせ"),
        ("Git", "分散型バージョン管理システム"),
        ("ブルーグリーンデプロイメント", "ダウンタイムなしでデプロイする手法"),
        ("カナリアリリース", "段階的にリリースする手法"),
        ("ローリングアップデート", "順次更新していくデプロイ手法"),
        ("インフラストラクチャの自動化", "インフラ管理を自動化する"),
        ("構成管理", "システム構成を一貫して管理する"),
        ("シークレット管理", "機密情報を安全に管理する"),
        ("サービスメッシュ", "マイクロサービス間の通信を管理する"),
        ("Istio", "人気のサービスメッシュプラットフォーム"),
        ("チャットOps", "チャットツールを通じた運用"),
        ("SRE", "Site Reliability Engineering - サイト信頼性エンジニアリング"),
        ("可観測性", "システムの内部状態を理解する能力"),
        ("インシデント管理", "障害対応のプロセス"),
        ("ポストモーテム", "障害後の振り返り分析"),
        ("GitOps", "Gitを使った運用管理手法")
    ]

    for topic, desc in devops_topics:
        qa_data.append({
            "category": "DevOps",
            "question": f"{topic}について説明してください",
            "answer": f"{topic}は、{desc}する概念・ツールです。"
        })

    # カテゴリー9: モバイル開発（30件）
    mobile_topics = [
        ("iOS開発", "Apple のモバイルプラットフォーム向け開発"),
        ("Android開発", "Google のモバイルプラットフォーム向け開発"),
        ("Swift", "iOS開発の主要言語"),
        ("Kotlin", "Android開発の推奨言語"),
        ("React Native", "クロスプラットフォームモバイル開発フレームワーク"),
        ("Flutter", "Googleのクロスプラットフォーム開発フレームワーク"),
        ("Xamarin", "Microsoftのクロスプラットフォーム開発ツール"),
        ("ネイティブアプリ", "特定のプラットフォーム専用のアプリ"),
        ("ハイブリッドアプリ", "Web技術とネイティブを組み合わせたアプリ"),
        ("プッシュ通知", "アプリからユーザーへの通知機能"),
        ("アプリ内購入", "アプリ内での課金機能"),
        ("モバイルUI/UX", "モバイル向けのユーザー体験設計"),
        ("レスポンシブデザイン", "様々な画面サイズに対応するデザイン"),
        ("オフライン機能", "ネットワークなしで動作する機能"),
        ("位置情報サービス", "GPSを使った位置ベースの機能"),
        ("モバイルセキュリティ", "モバイルアプリのセキュリティ対策"),
        ("アプリストア最適化", "App Store/Google Playでの可視性向上"),
        ("モバイルテスト", "モバイルアプリのテスト手法"),
        ("クラッシュレポート", "アプリのクラッシュを追跡する"),
        ("モバイル分析", "ユーザー行動の分析"),
        ("バックグラウンド処理", "アプリが前面にない時の処理"),
        ("ディープリンク", "アプリの特定画面への直接リンク"),
        ("生体認証", "指紋や顔認証による認証"),
        ("ARKit/ARCore", "拡張現実開発フレームワーク"),
        ("モバイルゲーム開発", "スマートフォン向けゲーム開発"),
        ("Progressive Web Apps", "Webアプリをネイティブ風に"),
        ("App Clips", "iOS の軽量アプリ体験"),
        ("Instant Apps", "Android のインストール不要アプリ"),
        ("モバイルDevOps", "モバイル開発のCI/CD"),
        ("コードサイニング", "アプリの正当性を証明する")
    ]

    for topic, desc in mobile_topics:
        qa_data.append({
            "category": "モバイル開発",
            "question": f"{topic}とは何ですか？",
            "answer": f"{topic}は、{desc}する技術・概念です。"
        })

    # カテゴリー10: ネットワーク（30件）
    network_topics = [
        ("TCP/IP", "インターネットの基本プロトコルスイート"),
        ("HTTP/HTTPS", "Web通信プロトコル"),
        ("DNS", "Domain Name System - ドメイン名解決システム"),
        ("ロードバランサー", "トラフィックを分散する装置"),
        ("ファイアウォール", "ネットワークセキュリティデバイス"),
        ("VPN", "仮想プライベートネットワーク"),
        ("CDN", "コンテンツ配信ネットワーク"),
        ("プロキシサーバー", "中継サーバー"),
        ("ルーター", "ネットワーク間をつなぐ装置"),
        ("スイッチ", "同一ネットワーク内の通信を制御"),
        ("OSI参照モデル", "ネットワーク通信の7層モデル"),
        ("IPv4/IPv6", "インターネットプロトコルのバージョン"),
        ("サブネット", "ネットワークの論理的な分割"),
        ("VLAN", "仮想的なLANセグメント"),
        ("NAT", "Network Address Translation - アドレス変換"),
        ("DHCP", "IPアドレスの自動割り当てプロトコル"),
        ("BGP", "インターネットのルーティングプロトコル"),
        ("SDN", "Software-Defined Networking"),
        ("帯域幅", "ネットワークの通信容量"),
        ("レイテンシ", "通信の遅延時間"),
        ("パケットロス", "データパケットの損失"),
        ("QoS", "Quality of Service - サービス品質"),
        ("ネットワークセグメンテーション", "ネットワークの分割"),
        ("ペリメーターセキュリティ", "境界防御"),
        ("DDoS対策", "分散サービス拒否攻撃への対策"),
        ("SSL/TLS", "暗号化通信プロトコル"),
        ("WebSocket", "双方向通信プロトコル"),
        ("REST API", "Webサービスのアーキテクチャスタイル"),
        ("gRPC", "高性能RPCフレームワーク"),
        ("マイクロセグメンテーション", "細かいネットワーク分割")
    ]

    for topic, desc in network_topics:
        qa_data.append({
            "category": "ネットワーク",
            "question": f"{topic}について教えてください",
            "answer": f"{topic}は、{desc}するネットワーク技術です。"
        })

    return qa_data

def generate_test_queries():
    """50件のテストクエリを生成"""
    test_queries = [
        # 機械学習・AI関連
        {"query": "ラベル付きデータで学習する手法", "expected_category": "機械学習・AI", "expected_match": "教師あり学習とは何ですか？"},
        {"query": "ディープラーニングについて教えて", "expected_category": "機械学習・AI", "expected_match": "深層学習とは何ですか？"},
        {"query": "報酬を使って学習するAI", "expected_category": "機械学習・AI", "expected_match": "強化学習とは何ですか？"},
        {"query": "複数のモデルを組み合わせる手法", "expected_category": "機械学習・AI", "expected_match": "アンサンブル学習とは何ですか？"},
        {"query": "AIの判断理由を説明できる技術", "expected_category": "機械学習・AI", "expected_match": "説明可能AIとは何ですか？"},

        # プログラミング言語
        {"query": "データサイエンスに最適な言語", "expected_category": "プログラミング言語", "expected_match": "Pythonの特徴は何ですか？"},
        {"query": "iOSアプリ開発の言語", "expected_category": "プログラミング言語", "expected_match": "Swiftの特徴は何ですか？"},
        {"query": "統計解析専門の言語", "expected_category": "プログラミング言語", "expected_match": "Rの特徴は何ですか？"},
        {"query": "メモリ安全なシステム言語", "expected_category": "プログラミング言語", "expected_match": "Rustの特徴は何ですか？"},
        {"query": "JavaScriptに型を追加した言語", "expected_category": "プログラミング言語", "expected_match": "TypeScriptの特徴は何ですか？"},

        # データベース
        {"query": "SQLを使わないデータベース", "expected_category": "データベース", "expected_match": "NoSQLデータベースとは何ですか？"},
        {"query": "メモリ上で動作する高速DB", "expected_category": "データベース", "expected_match": "インメモリデータベースとは何ですか？"},
        {"query": "関係性を表現するデータベース", "expected_category": "データベース", "expected_match": "グラフデータベースとは何ですか？"},
        {"query": "時間軸データ専門のDB", "expected_category": "データベース", "expected_match": "時系列データベースとは何ですか？"},
        {"query": "JSON形式でデータを保存するDB", "expected_category": "データベース", "expected_match": "ドキュメントデータベースとは何ですか？"},

        # クラウドサービス
        {"query": "Amazonのクラウドプラットフォーム", "expected_category": "クラウドサービス", "expected_match": "AWSについて説明してください"},
        {"query": "サーバー管理不要のアーキテクチャ", "expected_category": "クラウドサービス", "expected_match": "サーバーレスについて説明してください"},
        {"query": "アプリをパッケージ化する技術", "expected_category": "クラウドサービス", "expected_match": "コンテナについて説明してください"},
        {"query": "複数クラウドを使う戦略", "expected_category": "クラウドサービス", "expected_match": "マルチクラウドについて説明してください"},
        {"query": "インフラをコードで定義", "expected_category": "クラウドサービス", "expected_match": "Infrastructure as Codeについて説明してください"},

        # Web開発
        {"query": "Facebookが作ったJSライブラリ", "expected_category": "Web開発", "expected_match": "Reactとは何ですか？"},
        {"query": "単一ページで動作するWebアプリ", "expected_category": "Web開発", "expected_match": "SPAとは何ですか？"},
        {"query": "サーバーで事前にHTMLを生成", "expected_category": "Web開発", "expected_match": "SSRとは何ですか？"},
        {"query": "APIのクエリ言語", "expected_category": "Web開発", "expected_match": "GraphQLとは何ですか？"},
        {"query": "リアルタイム双方向通信", "expected_category": "Web開発", "expected_match": "WebSocketとは何ですか？"},

        # セキュリティ
        {"query": "2つの要素で認証する方法", "expected_category": "セキュリティ", "expected_match": "二要素認証について教えてください"},
        {"query": "SQLを悪用する攻撃", "expected_category": "セキュリティ", "expected_match": "SQLインジェクションについて教えてください"},
        {"query": "大量のトラフィックで攻撃", "expected_category": "セキュリティ", "expected_match": "DDoS攻撃について教えてください"},
        {"query": "データを人質に取るマルウェア", "expected_category": "セキュリティ", "expected_match": "ランサムウェアについて教えてください"},
        {"query": "何も信頼しないセキュリティモデル", "expected_category": "セキュリティ", "expected_match": "ゼロトラストセキュリティについて教えてください"},

        # データサイエンス
        {"query": "大規模データの処理技術", "expected_category": "データサイエンス", "expected_match": "ビッグデータとは何ですか？"},
        {"query": "データを視覚化する技術", "expected_category": "データサイエンス", "expected_match": "データビジュアライゼーションとは何ですか？"},
        {"query": "2つのバージョンを比較する実験", "expected_category": "データサイエンス", "expected_match": "A/Bテストとは何ですか？"},
        {"query": "データの前処理", "expected_category": "データサイエンス", "expected_match": "データクリーニングとは何ですか？"},
        {"query": "変数間の関係性を調べる", "expected_category": "データサイエンス", "expected_match": "相関分析とは何ですか？"},

        # DevOps
        {"query": "継続的にビルドとテストを行う", "expected_category": "DevOps", "expected_match": "CI/CDについて説明してください"},
        {"query": "コンテナを管理するプラットフォーム", "expected_category": "DevOps", "expected_match": "Kubernetesについて説明してください"},
        {"query": "インフラ構成管理ツール", "expected_category": "DevOps", "expected_match": "Terraformについて説明してください"},
        {"query": "システムの状態を可視化", "expected_category": "DevOps", "expected_match": "監視について説明してください"},
        {"query": "段階的にリリースする方法", "expected_category": "DevOps", "expected_match": "カナリアリリースについて説明してください"},

        # モバイル開発
        {"query": "クロスプラットフォームモバイル開発", "expected_category": "モバイル開発", "expected_match": "React Nativeとは何ですか？"},
        {"query": "Googleのモバイル開発フレームワーク", "expected_category": "モバイル開発", "expected_match": "Flutterとは何ですか？"},
        {"query": "アプリから通知を送る機能", "expected_category": "モバイル開発", "expected_match": "プッシュ通知とは何ですか？"},
        {"query": "オフラインでも動作する機能", "expected_category": "モバイル開発", "expected_match": "オフライン機能とは何ですか？"},
        {"query": "指紋や顔で認証する", "expected_category": "モバイル開発", "expected_match": "生体認証とは何ですか？"},

        # ネットワーク
        {"query": "インターネットの基本プロトコル", "expected_category": "ネットワーク", "expected_match": "TCP/IPについて教えてください"},
        {"query": "ドメイン名をIPアドレスに変換", "expected_category": "ネットワーク", "expected_match": "DNSについて教えてください"},
        {"query": "トラフィックを分散する装置", "expected_category": "ネットワーク", "expected_match": "ロードバランサーについて教えてください"},
        {"query": "コンテンツを地理的に配信", "expected_category": "ネットワーク", "expected_match": "CDNについて教えてください"},
        {"query": "通信の遅延時間", "expected_category": "ネットワーク", "expected_match": "レイテンシについて教えてください"}
    ]

    return test_queries

class RAGEvaluator:
    def __init__(self, embedding_model: str, dimensions: int = None):
        self.embedding_model = embedding_model
        self.dimensions = dimensions
        self.qa_data = generate_test_data()
        self.test_queries = generate_test_queries()
        self.embeddings = []
        self.df = None

    def generate_embeddings(self):
        """テストデータのEmbeddingsを生成（バッチ処理）"""
        texts = [f"質問: {qa['question']}\n回答: {qa['answer']}" for qa in self.qa_data]

        print(f"\nGenerating embeddings for {len(texts)} documents with {self.embedding_model}...")
        start_time = time.time()

        # バッチ処理（最大2048個ずつ）
        batch_size = 2048
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            print(f"  Processing batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}...")

            params = {
                "model": self.embedding_model,
                "input": batch
            }

            if self.dimensions and "text-embedding-3" in self.embedding_model:
                params["dimensions"] = self.dimensions

            response = client.embeddings.create(**params)
            batch_embeddings = [data.embedding for data in response.data]
            all_embeddings.extend(batch_embeddings)

        self.embeddings = all_embeddings
        elapsed_time = time.time() - start_time

        print(f"Generated {len(self.embeddings)} embeddings in {elapsed_time:.2f} seconds")
        print(f"  Average time per document: {elapsed_time/len(texts):.3f} seconds")

        # DataFrameに保存
        self.df = pd.DataFrame({
            'category': [qa['category'] for qa in self.qa_data],
            'question': [qa['question'] for qa in self.qa_data],
            'answer': [qa['answer'] for qa in self.qa_data],
            'text': texts,
            'embedding': self.embeddings
        })

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """検索を実行"""
        params = {
            "model": self.embedding_model,
            "input": query
        }

        if self.dimensions and "text-embedding-3" in self.embedding_model:
            params["dimensions"] = self.dimensions

        response = client.embeddings.create(**params)
        query_embedding = response.data[0].embedding

        # コサイン類似度を計算
        query_vec = np.array(query_embedding).reshape(1, -1)
        doc_vecs = np.array(self.df['embedding'].tolist())
        similarities = cosine_similarity(query_vec, doc_vecs)[0]

        # 結果を整理
        results = []
        for idx, sim in enumerate(similarities):
            results.append({
                'index': idx,
                'similarity': sim,
                'category': self.df.iloc[idx]['category'],
                'question': self.df.iloc[idx]['question'],
                'answer': self.df.iloc[idx]['answer']
            })

        results.sort(key=lambda x: x['similarity'], reverse=True)
        return results[:top_k]

    def evaluate(self, show_details: bool = False) -> Dict:
        """検索精度を評価"""
        print(f"\n{'='*80}")
        print(f"Evaluating {self.embedding_model}")
        if self.dimensions:
            print(f"Dimensions: {self.dimensions}")
        print(f"{'='*80}\n")

        correct_top1 = 0
        correct_top3 = 0
        correct_top5 = 0
        category_accuracy = 0
        total_queries = len(self.test_queries)

        # カテゴリ別の精度
        category_stats = {}

        detailed_results = []

        for i, test in enumerate(self.test_queries):
            query = test['query']
            expected_category = test['expected_category']
            expected_match = test['expected_match']

            # 検索実行
            results = self.search(query, top_k=5)

            # Top-1精度
            top1_match = results[0]['question'] == expected_match
            if top1_match:
                correct_top1 += 1

            # Top-3精度
            top3_match = any(r['question'] == expected_match for r in results[:3])
            if top3_match:
                correct_top3 += 1

            # Top-5精度
            top5_match = any(r['question'] == expected_match for r in results)
            if top5_match:
                correct_top5 += 1

            # カテゴリ精度
            if results[0]['category'] == expected_category:
                category_accuracy += 1

            # カテゴリ別統計
            if expected_category not in category_stats:
                category_stats[expected_category] = {'total': 0, 'correct': 0}
            category_stats[expected_category]['total'] += 1
            if top1_match:
                category_stats[expected_category]['correct'] += 1

            # 詳細結果を保存
            detailed_results.append({
                'query': query,
                'expected': expected_match,
                'top1_result': results[0]['question'],
                'top1_similarity': results[0]['similarity'],
                'top1_correct': top1_match,
                'category_correct': results[0]['category'] == expected_category
            })

            # 進捗表示
            if (i + 1) % 10 == 0:
                print(f"Progress: {i + 1}/{total_queries} queries evaluated...")

            # 詳細表示（オプション）
            if show_details and not top1_match:
                print(f"\nQuery: {query}")
                print(f"Expected: {expected_match}")
                print(f"Got: {results[0]['question']} (similarity: {results[0]['similarity']:.3f})")
                print(f"Correct answer rank: ", end="")
                for j, r in enumerate(results):
                    if r['question'] == expected_match:
                        print(f"#{j+1}")
                        break
                else:
                    print("Not in top-5")

        # サマリー
        summary = {
            'model': self.embedding_model,
            'dimensions': self.dimensions,
            'top1_accuracy': correct_top1 / total_queries,
            'top3_accuracy': correct_top3 / total_queries,
            'top5_accuracy': correct_top5 / total_queries,
            'category_accuracy': category_accuracy / total_queries,
            'category_stats': category_stats,
            'detailed_results': detailed_results
        }

        print(f"\n{'='*80}")
        print(f"SUMMARY - {self.embedding_model}")
        print(f"{'='*80}")
        print(f"Top-1 Accuracy: {summary['top1_accuracy']:.1%} ({correct_top1}/{total_queries})")
        print(f"Top-3 Accuracy: {summary['top3_accuracy']:.1%} ({correct_top3}/{total_queries})")
        print(f"Top-5 Accuracy: {summary['top5_accuracy']:.1%} ({correct_top5}/{total_queries})")
        print(f"Category Accuracy: {summary['category_accuracy']:.1%} ({category_accuracy}/{total_queries})")

        print(f"\nCategory-wise Top-1 Accuracy:")
        print(f"{'Category':<20} {'Accuracy':<10} {'Correct/Total'}")
        print("-" * 50)
        for cat, stats in sorted(category_stats.items()):
            acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
            print(f"{cat:<20} {acc:>6.1%}     {stats['correct']}/{stats['total']}")

        print(f"{'='*80}\n")

        return summary

def compare_models():
    """複数のモデルを比較"""
    configurations = [
        {"model": "text-embedding-3-small", "dimensions": None},
        {"model": "text-embedding-3-small", "dimensions": 1024},
        {"model": "text-embedding-3-large", "dimensions": None},
        {"model": "text-embedding-3-large", "dimensions": 1536},
        {"model": "text-embedding-ada-002", "dimensions": None},
    ]

    results = {}

    for config in configurations:
        model_name = config['model']
        dims = config['dimensions']
        key = f"{model_name}" + (f"-{dims}d" if dims else "")

        evaluator = RAGEvaluator(model_name, dims)
        evaluator.generate_embeddings()
        results[key] = evaluator.evaluate()

    # 比較結果を表示
    print("\n" + "="*100)
    print("MODEL COMPARISON SUMMARY (300 documents, 50 queries)")
    print("="*100)
    print(f"{'Model':<35} {'Dim':<6} {'Top-1':<10} {'Top-3':<10} {'Top-5':<10} {'Category':<10}")
    print("-"*100)

    for key, result in results.items():
        dims = result['dimensions'] if result['dimensions'] else 'default'
        print(f"{result['model']:<35} {dims:<6} {result['top1_accuracy']:>6.1%}     {result['top3_accuracy']:>6.1%}     {result['top5_accuracy']:>6.1%}     {result['category_accuracy']:>6.1%}")

    print("="*100)

# 実行例
if __name__ == "__main__":
    # 単一モデルの詳細評価
    print("Testing with 300 documents across 10 categories...")
    evaluator = RAGEvaluator("text-embedding-3-small")
    evaluator.generate_embeddings()
    evaluator.evaluate(show_details=True)